# LangChain Hands-on 09 — Vector Stores
**Trainer:** Ajeetkumar

**What we will practise**
- What is a vector database?
- InMemoryVectorStore
- FAISS (save / load)
- Chroma (persistent)
- add, search, search with score, delete
- Metadata filtering

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu langchain-chroma

# What is a Vector Store?
- A database that stores **embeddings + text + metadata**.
- Finds the most **similar** chunks to a question very fast.

```
chunks ──embed──▶ [vector store] ◀──embed── question
                        │
                 top-k similar chunks
```
| Store | Type | Good for |
|---|---|---|
| InMemory | RAM | learning, small demos |
| FAISS | local file | fast, offline |
| Chroma | local DB | persistent, metadata filters |
| Pinecone / Qdrant / Weaviate / pgvector / Azure AI Search | cloud | production |

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
docs = [
    Document(page_content="Report a lost card within 3 days for zero liability.", metadata={"topic": "cards", "id": 1}),
    Document(page_content="Card-not-present payments above Rs 5,000 need OTP.", metadata={"topic": "cards", "id": 2}),
    Document(page_content="UPI limit for a new payee is Rs 25,000 in the first 24 hours.", metadata={"topic": "upi", "id": 3}),
    Document(page_content="Never share your UPI PIN with anyone.", metadata={"topic": "upi", "id": 4}),
    Document(page_content="Video KYC is available from 9 AM to 9 PM.", metadata={"topic": "kyc", "id": 5}),
    Document(page_content="Home loan interest starts at 8.5% per year.", metadata={"topic": "loans", "id": 6}),
]

## 1. InMemoryVectorStore

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

In [ ]:
vector_store = InMemoryVectorStore.from_documents(docs, embeddings)

In [ ]:
results = vector_store.similarity_search("I lost my card, what should I do?", k=2)
for r in results:
    print(r.metadata, "→", r.page_content)

### Search with score (higher = more similar for this store)

In [ ]:
for doc, score in vector_store.similarity_search_with_score("Is it safe to tell my PIN to bank staff?", k=3):
    print(round(score, 3), "→", doc.page_content)

## 2. FAISS — fast and can be saved to disk

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
faiss_store = FAISS.from_documents(docs, embeddings)

In [ ]:
faiss_store.similarity_search("loan interest rate", k=1)

In [ ]:
# FAISS score = distance → LOWER is more similar
for doc, score in faiss_store.similarity_search_with_score("loan interest rate", k=3):
    print(round(score, 3), "→", doc.page_content)

In [ ]:
faiss_store.save_local("faiss_bank_index")

In [ ]:
loaded = FAISS.load_local("faiss_bank_index", embeddings, allow_dangerous_deserialization=True)
loaded.similarity_search("KYC timing", k=1)

### Add more documents later

In [ ]:
faiss_store.add_documents([Document(page_content="Fixed deposit rate is 7.1% for 1 year.", metadata={"topic": "deposits", "id": 7})])
faiss_store.similarity_search("FD interest", k=1)

## 3. Chroma — persistent DB with metadata filters

In [ ]:
from langchain_chroma import Chroma

In [ ]:
chroma_store = Chroma.from_documents(docs, embeddings, collection_name="bank_faq", persist_directory="./chroma_db")

In [ ]:
chroma_store.similarity_search("payment limit", k=2)

### Metadata filter — search only inside one topic

In [ ]:
chroma_store.similarity_search("limit", k=2, filter={"topic": "upi"})

In [ ]:
chroma_store.similarity_search("limit", k=2, filter={"topic": "cards"})

### Add with ids, then delete

In [ ]:
chroma_store.add_documents([Document(page_content="Temporary offer: zero fee on credit cards.", metadata={"topic": "offers"})], ids=["offer-1"])
chroma_store.similarity_search("offer", k=1)

In [ ]:
chroma_store.delete(ids=["offer-1"])
chroma_store.similarity_search("offer", k=1)

### Re-open the saved Chroma DB

In [ ]:
reopened = Chroma(collection_name="bank_faq", embedding_function=embeddings, persist_directory="./chroma_db")
reopened.similarity_search("video KYC", k=1)

## 4. MMR search — relevant AND diverse results

In [ ]:
chroma_store.max_marginal_relevance_search("card rules", k=2, fetch_k=6)

# Benefits of Vector Stores
- **Semantic search** (meaning, not just keywords)
- **Fast** over millions of chunks
- **Metadata filters** (by user, date, department)
- Core of every **RAG** system

## 🧪 Practice
1. Load a PDF, split it, store in FAISS, save and reload it.
2. Store 20 product descriptions in Chroma with `category` metadata and filter by category.
3. Compare FAISS vs InMemory scores for the same question.